In [3]:
import pandas as pd
import numpy as np
from prefixspan import PrefixSpan

from src.algo.gsp import GSPAlgo
from src.data.base_data import Data

In [4]:
# Load raw cancer dataset using Data class
data_obj = Data("cancer")

# Initialize GSP algorithm
gsp = GSPAlgo(data_obj)


In [5]:
# Helper Function
def convert_dataframe_to_sequences(df, top_k=3):
    """
    Convert DataFrame from GSPAlgo (with feature_1_name, feature_1_value, etc.) 
    to sequences format for PrefixSpan.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with columns: patient_id, diagnosis, feature_1_name, feature_1_value, ...
    top_k : int
        Number of top-K features to include (K for feature transformation)
    Returns:
    --------
    tuple: (malignant_sequences, benign_sequences)
        Lists of sequences for malignant and benign patients
    """
    malignant_sequences = []
    benign_sequences = []
    
    for idx, row in df.iterrows():
        # Create sequence from feature_1, feature_2, ..., feature_K
        sequence = []
        for i in range(1, top_k + 1):
            name_col = f'feature_{i}_name'
            value_col = f'feature_{i}_value'
            
            if name_col in row.index and value_col in row.index:
                name = row[name_col]
                value = row[value_col]
                
                # Skip empty features
                if pd.notna(name) and pd.notna(value) and name != '' and value != '':
                    item = f"{value}_{name}"
                    # Each itemset is a tuple
                    sequence.append((item,))
        
        # Add to appropriate list
        if row['diagnosis'] == 'M':
            malignant_sequences.append(sequence)
        elif row['diagnosis'] == 'B':
            benign_sequences.append(sequence)
    
    return malignant_sequences, benign_sequences

In [6]:
# Helper Function
def mine_patterns(sequences, maxlen=2, min_sup=0.02):
    """
    Mine frequent sequential patterns using PrefixSpan.
    
    Parameters:
    -----------
    sequences : list
        List of sequences (each sequence is a list of tuple itemsets)
    maxlen : int
        Maximum length of patterns to mine
    min_sup : float
        Minimum support as a fraction (e.g., 0.02 = 2%)
    
    Returns:
    --------
    list: List of (support, pattern) tuples
    """
    if not sequences:
        return []
    
    # Calculate support count from the fraction
    support_count = max(1, int(min_sup * len(sequences)))
    
    try:
        ps = PrefixSpan(sequences)
        ps.minlen = 2  # Allow patterns of length 2 and above. Otherwise, it will not be a "sequence"
        
        # Find frequent patterns based on support count
        frequent_patterns = ps.frequent(support_count)
        
        # Filter patterns by maxlen (pattern length = number of itemsets)
        filtered_patterns = [
            (support, pattern) for support, pattern in frequent_patterns
            if len(pattern) <= maxlen
        ]
        
        return filtered_patterns
    except Exception as e:
        print(f"Error mining patterns: {e}")
        return []

In [7]:
# Helper Function
def process_patterns_to_df(patterns_list, total_sequences):
    """Converts the (support, pattern) list into a DataFrame."""
    if not patterns_list or total_sequences == 0:
        return pd.DataFrame(columns=['pattern', 'support_count', 'support_percent'])
    
    # Convert list of tuples to DataFrame
    df = pd.DataFrame(patterns_list, columns=['support_count', 'pattern'])
    
    # Calculate support percentage
    df['support_percent'] = df['support_count'] / total_sequences
    
    # Convert the pattern (which is a list) to a string so we can merge on it
    df['pattern'] = df['pattern'].astype(str)
    
    return df

In [8]:
# Step 1: Parameter tuning. For each strategy, we will try different K, L, and min_sup values to find the best combination.
# Define parameter grid
strategies = ['uniform', 'quantile', 'kmeans']
K_values = [3, 5, 10]
L_values = [3, 5, 10]
min_sup_values = [0.15, 0.10, 0.05]

# Store results
results = []

print("Starting Parameter Tuning Process...")
print(f"Total combinations: {len(strategies)} strategies × {len(K_values)} K values × {len(L_values)} L values × {len(min_sup_values)} min_sup values")
print(f"= {len(strategies) * len(K_values) * len(L_values) * len(min_sup_values)} total runs\n")

for strategy in strategies:
    
    for K in K_values:
        # Generate sequences with the specific K value (top_k=K for feature transformation)
        sequences_df =  gsp._generate_sequences_for_strategy(strategy, top_k=K)
        sequences_mal, sequences_ben = convert_dataframe_to_sequences(sequences_df, top_k=K)
        
        # Mine patterns using PrefixSpan
        for min_sup in min_sup_values:
            for L in L_values:
                # Mine patterns for malignant and benign
                # maxlen=L is for pattern mining (maximum pattern length)
                patterns_m = mine_patterns(sequences_mal, maxlen=L, min_sup=min_sup)
                patterns_b = mine_patterns(sequences_ben, maxlen=L, min_sup=min_sup)
                
                # Record results
                results.append({
                    'strategy': strategy,
                    'top-K features K': K,
                    'min_sup': min_sup,
                    'max sequence length L': L,
                    'malignant_patterns_length': len(patterns_m),
                    'benign_patterns_length': len(patterns_b),
                    'total_patterns': len(patterns_m) + len(patterns_b)
                })

# Create results DataFrame
results_df = pd.DataFrame(results)

# Save the result to csv
results_df.to_csv('02_parameter_tuning_summary.csv',index=False)

print(f"\nCompleted {len(results)} runs! Detailed summary is saved to '02_parameter_tuning_summary.csv'")
print("The overview of the results is shown below:")
results_df



Starting Parameter Tuning Process...
Total combinations: 3 strategies × 3 K values × 3 L values × 3 min_sup values
= 81 total runs


Completed 81 runs! Detailed summary is saved to '02_parameter_tuning_summary.csv'
The overview of the results is shown below:


,strategy,top-K features K,min_sup,max sequence length L,malignant_patterns_length,benign_patterns_length,total_patterns
0,uniform,3,0.15,3,0,0,0
1,uniform,3,0.15,5,0,0,0
2,uniform,3,0.15,10,0,0,0
3,uniform,3,0.10,3,0,0,0
4,uniform,3,0.10,5,0,0,0
...,...,...,...,...,...,...,...
76,kmeans,10,0.10,5,38,51,89
77,kmeans,10,0.10,10,38,51,89
78,kmeans,10,0.05,3,266,286,552
79,kmeans,10,0.05,5,272,295,567


In [9]:
# Tuned parameters was selected:
TUNED_TOP_K = 10
TUNED_MIN_SUP = 0.1
TUNED_MAX_LEN = 10
STRATEGIES = ['uniform', 'quantile', 'kmeans']



# Step 2: Mining cancer feature patterns using the tuned parameters.
# This list will hold the DataFrames from all 3 strategies
all_results = []

print("--- Starting Analysis Loop for All Strategies ---")

for strategy in STRATEGIES:
    print(f"\nProcessing strategy: '{strategy}'...")
    
    # Step 2.1: Generate sequences using the tuned K
    sequences_df = gsp._generate_sequences_for_strategy(strategy, top_k=TUNED_TOP_K)

    # Step 2.2: Convert the DataFrame to sequence lists
    sequences_mal, sequences_ben = convert_dataframe_to_sequences(sequences_df, top_k=TUNED_TOP_K)
    total_mal_seqs = len(sequences_mal)
    total_ben_seqs = len(sequences_ben)

    # Step 2.3: Mine patterns from each list
    patterns_m = mine_patterns(sequences_mal, maxlen=TUNED_MAX_LEN, min_sup=TUNED_MIN_SUP)
    patterns_b = mine_patterns(sequences_ben, maxlen=TUNED_MAX_LEN, min_sup=TUNED_MIN_SUP)
    print(f"  Found {len(patterns_m)} frequent patterns for malignant and {len(patterns_b)} frequent patterns for benign.")

    # Step 2.4: Process into DataFrames
    df_m = process_patterns_to_df(patterns_m, total_mal_seqs)
    df_b = process_patterns_to_df(patterns_b, total_ben_seqs)

    # Step 2.5: Merge and Calculate Rates
    df_compare = pd.merge(
        df_m, 
        df_b, 
        on='pattern', 
        how='outer',
        suffixes=('_m', '_b')
    )
    df_compare = df_compare.fillna(0)

    # Step 2.6: Integrate Growth Rate and Contrast Rate
    df_compare['GR_Malignant'] = df_compare['support_percent_m'] / df_compare['support_percent_b']
    df_compare['GR_Benign'] = df_compare['support_percent_b'] / df_compare['support_percent_m']
    df_compare['Contrast_Rate'] = df_compare[['GR_Malignant', 'GR_Benign']].max(axis=1)
    df_compare['Contrast_Rate'] = df_compare['Contrast_Rate'].replace(np.nan, np.inf)

    # Step 2.7: Add pattern_length and strategy columns
    df_compare['pattern_length'] = df_compare['pattern'].str.count(r'\(')
    df_compare['strategy'] = strategy
    
    # Step 2.8: Add this strategy's results to our main list
    all_results.append(df_compare)

print("\n--- Analysis Loop Complete ---")

# Combine all results into one single DataFrame
all_results_df = pd.concat(all_results, ignore_index=True)

# Reorder columns to match your request
final_columns = [
    'strategy', 
    'pattern', 
    'pattern_length', 
    'support_percent_m', 
    'support_percent_b',
    'support_count_m',
    'support_count_b',
    'GR_Malignant', 
    'GR_Benign', 
    'Contrast_Rate'
]
all_results_df = all_results_df[final_columns]
all_results_df.to_csv('02_all_results_df.csv',index=False)

print(f"Created final DataFrame with {len(all_results_df)} total frequent patterns from all strategies.")
print("The detailed results are saved to '02_all_results_df.csv'")
print("The overview of the results is shown below:")
all_results_df

--- Starting Analysis Loop for All Strategies ---

Processing strategy: 'uniform'...
  Found 30 frequent patterns for malignant and 55 frequent patterns for benign.

Processing strategy: 'quantile'...
  Found 95 frequent patterns for malignant and 46 frequent patterns for benign.

Processing strategy: 'kmeans'...
  Found 38 frequent patterns for malignant and 51 frequent patterns for benign.

--- Analysis Loop Complete ---
Created final DataFrame with 315 total frequent patterns from all strategies.
The detailed results are saved to '02_all_results_df.csv'
The overview of the results is shown below:


,strategy,pattern,pattern_length,support_percent_m,support_percent_b,support_count_m,support_count_b,GR_Malignant,GR_Benign,Contrast_Rate
0,uniform,"[('high_concave points_worst',), ('medium_conc...",2,0.108491,0.000000,23.0,0.0,inf,0.0,inf
1,uniform,"[('high_concave points_worst',), ('medium_conc...",2,0.099057,0.000000,21.0,0.0,inf,0.0,inf
2,uniform,"[('low_compactness_mean',), ('low_compactness_...",2,0.000000,0.098039,0.0,35.0,0.0,inf,inf
3,uniform,"[('low_compactness_mean',), ('low_compactness_...",2,0.000000,0.134454,0.0,48.0,0.0,inf,inf
4,uniform,"[('low_compactness_mean',), ('low_concavity_me...",2,0.000000,0.106443,0.0,38.0,0.0,inf,inf
...,...,...,...,...,...,...,...,...,...,...
310,kmeans,"[('medium_area_worst',), ('high_perimeter_mean...",2,0.103774,0.000000,22.0,0.0,inf,0.0,inf
311,kmeans,"[('medium_area_worst',), ('high_perimeter_wors...",2,0.117925,0.000000,25.0,0.0,inf,0.0,inf
312,kmeans,"[('medium_perimeter_se',), ('medium_area_se',)]",2,0.103774,0.000000,22.0,0.0,inf,0.0,inf
313,kmeans,"[('medium_radius_se',), ('medium_area_se',)]",2,0.160377,0.000000,34.0,0.0,inf,0.0,inf


In [10]:
# Step 3: Pattern Interpretation
# Step 3.1: Ranking by Support Percentage

# --- 1. Ranking by Support Percentage (Regardless of Contrast) ---

TOP_N_RANKS = 5
ranks = list(range(1, TOP_N_RANKS + 1))

# --- Malignant Table ---
print("--- Top 5 Patterns by Malignant Support % ---")
df_support_m = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    # 1. Filter for strategy
    strategy_df = all_results_df[all_results_df['strategy'] == strategy]
    
    # 2. Sort by malignant support
    sorted_df = strategy_df.sort_values(by='support_percent_m', ascending=False)
    
    # 3. Get just the pattern string
    formatted_list = [row['pattern'] for _, row in sorted_df.head(TOP_N_RANKS).iterrows()]
    
    # 4. Add to our table
    df_support_m[strategy] = formatted_list

display(df_support_m)

# --- Benign Table ---
print("\n--- Top 5 Patterns by Benign Support % ---")
df_support_b = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    # 1. Filter
    strategy_df = all_results_df[all_results_df['strategy'] == strategy]
    
    # 2. Sort by benign support
    sorted_df = strategy_df.sort_values(by='support_percent_b', ascending=False)
    
    # 3. Get just the pattern string
    formatted_list = [row['pattern'] for _, row in sorted_df.head(TOP_N_RANKS).iterrows()]
    
    # 4. Add
    df_support_b[strategy] = formatted_list
    
display(df_support_b)

--- Top 5 Patterns by Malignant Support % ---


,uniform,quantile,kmeans
Rank,,,
1,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_radius_mean',), ('high_perimeter_mean',)]","[('high_radius_mean',), ('high_perimeter_mean',)]"
2,"[('medium_radius_worst',), ('medium_perimeter_...","[('high_radius_worst',), ('high_perimeter_wors...","[('high_radius_mean',), ('medium_area_mean',)]"
3,"[('medium_radius_mean',), ('medium_area_mean',)]","[('high_radius_worst',), ('high_perimeter_mean...","[('high_radius_worst',), ('high_perimeter_wors..."
4,"[('medium_perimeter_mean',), ('medium_perimete...","[('high_radius_mean',), ('high_area_mean',)]","[('high_radius_worst',), ('high_perimeter_mean..."
5,"[('medium_radius_mean',), ('medium_perimeter_w...","[('high_area_mean',), ('high_perimeter_mean',)]","[('high_radius_worst',), ('medium_area_worst',)]"



--- Top 5 Patterns by Benign Support % ---


,uniform,quantile,kmeans
Rank,,,
1,"[('low_radius_mean',), ('low_perimeter_mean',)]","[('low_radius_mean',), ('low_perimeter_mean',)]","[('low_radius_mean',), ('low_perimeter_mean',)]"
2,"[('low_radius_mean',), ('low_area_mean',)]","[('low_radius_mean',), ('low_area_mean',)]","[('low_radius_mean',), ('low_area_mean',)]"
3,"[('low_radius_mean',), ('low_perimeter_worst',)]","[('low_perimeter_mean',), ('low_area_mean',)]","[('low_perimeter_mean',), ('low_area_mean',)]"
4,"[('low_perimeter_mean',), ('low_area_mean',)]","[('low_radius_mean',), ('low_perimeter_worst',)]","[('low_radius_mean',), ('low_perimeter_worst',)]"
5,"[('low_perimeter_mean',), ('low_perimeter_wors...","[('low_perimeter_mean',), ('low_perimeter_wors...","[('low_perimeter_mean',), ('low_perimeter_wors..."


In [13]:
# Step 3.2: Ranking by Contrast Rate

TOP_N_RANKS = 5
ranks = list(range(1, TOP_N_RANKS + 1))

# --- Malignant-Leaning Table ---
print("--- Top 5 Malignant Patterns by Contrast Rate ---")
df_contrast_m = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    strategy_df = all_results_df[all_results_df['strategy'] == strategy]
    
    # Sort by Contrast_Rate (desc), with malignant support as a tie-breaker
    sorted_df = strategy_df.sort_values(
        by=['Contrast_Rate', 'support_percent_m'], 
        ascending=[False, False]
    )
    
    # Format the pattern and CR
    formatted_list = []
    for _, row in sorted_df.head(TOP_N_RANKS).iterrows():
        cr_str = "inf" if np.isinf(row['Contrast_Rate']) else f"{row['Contrast_Rate']:.2f}"
        formatted_list.append(
            f"{row['pattern']} (CR: {cr_str})"
        )
    
    df_contrast_m[strategy] = formatted_list

display(df_contrast_m)

# --- Benign-Leaning Table ---
print("\n--- Top 5 Benign Patterns by Contrast Rate ---")
df_contrast_b = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    strategy_df = all_results_df[all_results_df['strategy'] == strategy]
    
    # Sort by Contrast_Rate (asc), with benign support as a tie-breaker
    sorted_df = strategy_df.sort_values(
        by=['Contrast_Rate', 'support_percent_b'], 
        ascending=[True, False]
    )
    
    # Format
    formatted_list = []
    for _, row in sorted_df.head(TOP_N_RANKS).iterrows():
        cr_str = f"{row['Contrast_Rate']:.2f}"
        formatted_list.append(
            f"{row['pattern']} (CR: {cr_str})"
        )
        
    df_contrast_b[strategy] = formatted_list

display(df_contrast_b)

--- Top 5 Malignant Patterns by Contrast Rate ---


,uniform,quantile,kmeans
Rank,,,
1,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_radius_mean',), ('high_perimeter_mean'...","[('high_radius_mean',), ('high_perimeter_mean'..."
2,"[('medium_radius_worst',), ('medium_perimeter_...","[('high_radius_worst',), ('high_perimeter_wors...","[('high_radius_mean',), ('medium_area_mean',)]..."
3,"[('medium_radius_mean',), ('medium_area_mean',...","[('high_radius_worst',), ('high_perimeter_mean...","[('high_radius_worst',), ('high_perimeter_wors..."
4,"[('medium_perimeter_mean',), ('medium_perimete...","[('high_area_mean',), ('high_perimeter_mean',)...","[('high_radius_worst',), ('high_perimeter_mean..."
5,"[('medium_radius_mean',), ('medium_perimeter_w...","[('high_radius_mean',), ('high_area_mean',)] (...","[('high_radius_worst',), ('medium_area_worst',..."



--- Top 5 Benign Patterns by Contrast Rate ---


,uniform,quantile,kmeans
Rank,,,
1,"[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_area_mean',)] (CR...","[('low_radius_mean',), ('low_perimeter_mean',)..."
2,"[('low_radius_mean',), ('low_area_mean',)] (CR...","[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_area_mean',)] (CR..."
3,"[('low_perimeter_mean',), ('low_area_mean',)] ...","[('low_perimeter_mean',), ('low_area_mean',)] ...","[('low_perimeter_mean',), ('low_area_mean',)] ..."
4,"[('low_radius_mean',), ('low_perimeter_worst',...","[('low_radius_mean',), ('low_perimeter_worst',...","[('low_radius_mean',), ('low_perimeter_worst',..."
5,"[('low_perimeter_mean',), ('low_perimeter_wors...","[('low_perimeter_mean',), ('low_perimeter_wors...","[('low_perimeter_mean',), ('low_perimeter_wors..."


In [16]:
# Step 3.3: Ranking by Specificity (Contrast Rate == infinity Only)

TOP_N_RANKS = 5
ranks = list(range(1, TOP_N_RANKS + 1))

# First, filter for only exclusive patterns (CR == inf)
exclusive_patterns_df = all_results_df[all_results_df['Contrast_Rate'] == np.inf].copy()

# --- Exclusive Malignant Table ---
print("--- Top 5 Most SPECIFIC Exclusive Malignant Patterns ---")
df_spec_m = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    # Filter for strategy AND exclusive malignant patterns
    strategy_df = exclusive_patterns_df[
        (exclusive_patterns_df['strategy'] == strategy) &
        (exclusive_patterns_df['support_percent_m'] > 0)
    ]
    
    # Sort by Length (desc), then Support (desc)
    sorted_df = strategy_df.sort_values(
        by=['pattern_length', 'support_percent_m'],
        ascending=[False, False]
    )
    
    # Format
    formatted_list = [
        f"{row['pattern']} (Len: {row['pattern_length']})" 
        for _, row in sorted_df.head(TOP_N_RANKS).iterrows()
    ]
    
    # Pad the list if fewer than 15 patterns were found
    if len(formatted_list) < TOP_N_RANKS:
        formatted_list.extend(['-'] * (TOP_N_RANKS - len(formatted_list)))
    
    df_spec_m[strategy] = formatted_list

print("(Ranked by Length)")
display(df_spec_m)

# --- Exclusive Benign Table ---
print("\n--- Top 5 Most SPECIFIC Exclusive Benign Patterns ---")
df_spec_b = pd.DataFrame({'Rank': ranks}).set_index('Rank')

for strategy in STRATEGIES:
    # Filter for strategy AND exclusive benign patterns
    strategy_df = exclusive_patterns_df[
        (exclusive_patterns_df['strategy'] == strategy) &
        (exclusive_patterns_df['support_percent_b'] > 0)
    ]
    
    # Sort by Length (desc), then Support (desc)
    sorted_df = strategy_df.sort_values(
        by=['pattern_length', 'support_percent_b'],
        ascending=[False, False]
    )
    
    # Format
    formatted_list = [
        f"{row['pattern']} (Len: {row['pattern_length']})" 
        for _, row in sorted_df.head(TOP_N_RANKS).iterrows()
    ]
    
    # Pad
    if len(formatted_list) < TOP_N_RANKS:
        formatted_list.extend(['-'] * (TOP_N_RANKS - len(formatted_list)))
        
    df_spec_b[strategy] = formatted_list

display(df_spec_b)

--- Top 5 Most SPECIFIC Exclusive Malignant Patterns ---
(Ranked by Length)


,uniform,quantile,kmeans
Rank,,,
1,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_radius_mean',), ('high_perimeter_mean'...","[('high_radius_mean',), ('high_perimeter_mean'..."
2,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_radius_mean',), ('high_perimeter_mean'...","[('high_radius_worst',), ('high_radius_mean',)..."
3,"[('medium_radius_mean',), ('medium_radius_wors...","[('high_radius_mean',), ('high_perimeter_mean'...","[('high_radius_mean',), ('high_perimeter_mean'..."
4,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_radius_mean',), ('high_radius_worst',)...","[('high_radius_mean',), ('medium_area_mean',)]..."
5,"[('medium_radius_mean',), ('medium_perimeter_m...","[('high_area_worst',), ('high_radius_mean',), ...","[('high_radius_worst',), ('high_perimeter_wors..."



--- Top 5 Most SPECIFIC Exclusive Benign Patterns ---


,uniform,quantile,kmeans
Rank,,,
1,"[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)..."
2,"[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)..."
3,"[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)...","[('low_radius_mean',), ('low_perimeter_mean',)..."
4,"[('low_radius_mean',), ('low_perimeter_worst',...","[('low_radius_mean',), ('low_perimeter_worst',...","[('low_radius_mean',), ('low_perimeter_worst',..."
5,"[('low_radius_mean',), ('low_radius_worst',), ...","[('low_radius_mean',), ('low_radius_worst',), ...","[('low_radius_mean',), ('low_radius_worst',), ..."
